<a href="https://colab.research.google.com/github/loukaBl/Train-and-evaluate-model-Iris-dataset-/blob/main/Train_and_evaluate_model_(Iris_dataset).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

In [2]:
PALETTE   = ["#6C63FF", "#FF6584", "#43C59E"]
BG        = "#F8F9FC"
DARK      = "#1E1E2E"
sns.set_theme(style="whitegrid", palette=PALETTE)
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG,
    "font.family": "DejaVu Sans", "axes.titlesize": 13,
    "axes.labelsize": 11,
})

In [3]:
iris   = load_iris()
df     = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print("=" * 55)
print("  IRIS DATASET — MACHINE LEARNING WALKTHROUGH")
print("=" * 55)

  IRIS DATASET — MACHINE LEARNING WALKTHROUGH


In [4]:
print(f"\n Shape      : {df.shape}")
print(f" Features   : {list(iris.feature_names)}")
print(f" Classes    : {list(iris.target_names)}")
print(f"\n Class distribution:\n{df['species'].value_counts().to_string()}")
print(f"\nFirst 5 rows:\n{df.head().to_string(index=False)}")
print(f"\nDescriptive stats:\n{df.describe().round(2).to_string()}")


 Shape      : (150, 5)
 Features   : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
 Classes    : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

 Class distribution:
species
setosa        50
versicolor    50
virginica     50

First 5 rows:
 sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm) species
               5.1               3.5                1.4               0.2  setosa
               4.9               3.0                1.4               0.2  setosa
               4.7               3.2                1.3               0.2  setosa
               4.6               3.1                1.5               0.2  setosa
               5.0               3.6                1.4               0.2  setosa

Descriptive stats:
       sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
count             150.00            150.00             150.00            150.00
mean                5.84        

In [5]:
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\n  Train size : {X_train.shape[0]}  |  Test size : {X_test.shape[0]}")


  Train size : 120  |  Test size : 30


In [6]:
scaler  = StandardScaler().fit(X_train)
Xs_tr   = scaler.transform(X_train)
Xs_te   = scaler.transform(X_test)

In [7]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(Xs_tr, y_train)
y_pred_knn = knn.predict(Xs_te)

In [8]:
lr  = LogisticRegression(max_iter=200, random_state=42)
lr.fit(Xs_tr, y_train)
y_pred_lr = lr.predict(Xs_te)

In [9]:
for label, y_pred, model in [
    ("KNN (k=5)", y_pred_knn, knn),
    ("Logistic Regression", y_pred_lr, lr),
]:
    acc = accuracy_score(y_test, y_pred)
    print(f"\n{'─'*45}")
    print(f"  {label}  —  Accuracy: {acc*100:.1f}%")
    print(f"{'─'*45}")
    print(classification_report(y_test, y_pred,
                                target_names=iris.target_names))


─────────────────────────────────────────────
  KNN (k=5)  —  Accuracy: 100.0%
─────────────────────────────────────────────
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


─────────────────────────────────────────────
  Logistic Regression  —  Accuracy: 100.0%
─────────────────────────────────────────────
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.

In [11]:
import os

fig1, axes = plt.subplots(2, 2, figsize=(12, 9))
fig1.suptitle("Iris Dataset — Feature Overview by Species",
              fontsize=15, fontweight="bold", color=DARK, y=1.01)

feature_pairs = [
    ("sepal length (cm)", "sepal width (cm)"),
    ("petal length (cm)", "petal width (cm)"),
    ("sepal length (cm)", "petal length (cm)"),
    ("sepal width (cm)",  "petal width (cm)"),
]
for ax, (fx, fy) in zip(axes.flat, feature_pairs):
    for i, (sp, col) in enumerate(zip(iris.target_names, PALETTE)):
        mask = df["species"] == sp
        ax.scatter(df.loc[mask, fx], df.loc[mask, fy],
                   color=col, alpha=0.75, edgecolors="white",
                   linewidths=0.5, s=60, label=sp)
    ax.set_xlabel(fx); ax.set_ylabel(fy)
    ax.set_title(f"{fx.split()[0].title()} vs {fy.split()[0].title()}")

handles = [mpatches.Patch(color=c, label=n)
           for c, n in zip(PALETTE, iris.target_names)]
fig1.legend(handles=handles, loc="lower center",
            ncol=3, frameon=False, fontsize=10,
            bbox_to_anchor=(0.5, -0.03))
fig1.tight_layout()

# Create the directory if it doesn't exist
os.makedirs('/mnt/user-data/outputs/', exist_ok=True)
fig1.savefig("/mnt/user-data/outputs/fig1_eda_scatter.png",
             dpi=150, bbox_inches="tight", facecolor=BG)
plt.close(fig1)


In [12]:
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fig2.suptitle("Confusion Matrices", fontsize=15,
              fontweight="bold", color=DARK)

for ax, y_pred, title in [
    (ax1, y_pred_knn, f"KNN (k=5) — {accuracy_score(y_test, y_pred_knn)*100:.1f}% acc"),
    (ax2, y_pred_lr,  f"Logistic Regression — {accuracy_score(y_test, y_pred_lr)*100:.1f}% acc"),
]:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="RdPu",
                xticklabels=iris.target_names,
                yticklabels=iris.target_names,
                linewidths=1, linecolor="white",
                ax=ax, cbar=False, annot_kws={"size": 14})
    ax.set_title(title, pad=10)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual", fontsize=11)

fig2.tight_layout()
fig2.savefig("/mnt/user-data/outputs/fig2_confusion_matrices.png",
             dpi=150, bbox_inches="tight", facecolor=BG)
plt.close(fig2)

In [18]:
feat_idx = [2, 3]   # petal length, petal width
feat_names = [iris.feature_names[i] for i in feat_idx]

X2 = iris.data[:, feat_idx]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42
)
sc2  = StandardScaler().fit(X2_train)
X2s_tr = sc2.transform(X2_train)
X2s_te = sc2.transform(X2_test)

knn2 = KNeighborsClassifier(n_neighbors=5).fit(X2s_tr, y2_train)
lr2  = LogisticRegression(max_iter=200, random_state=42).fit(X2s_tr, y2_train)

x_min, x_max = X2s_tr[:, 0].min() - 0.6, X2s_tr[:, 0].max() + 0.6
y_min, y_max = X2s_tr[:, 1].min() - 0.6, X2s_tr[:, 1].max() + 0.6
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))

from matplotlib.colors import ListedColormap
cmap_bg  = ListedColormap(["#D8D5FF", "#FFCCD8", "#B8F0E0"])
cmap_dot = ListedColormap(PALETTE)

fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig3.suptitle("Decision Boundaries — Petal Features (Scaled)",
              fontsize=15, fontweight="bold", color=DARK)

for ax, model, title in [
    (ax1, knn2, "KNN (k=5)"),
    (ax2, lr2,  "Logistic Regression"),
]:
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.55, cmap=cmap_bg)
    sc = ax.scatter(X2s_te[:, 0], X2s_te[:, 1],
                    c=y2_test, cmap=cmap_dot,
                    edgecolors="white", linewidths=0.8, s=80, zorder=3)
    preds = model.predict(X2s_te)
    wrong = preds != y2_test
    ax.scatter(X2s_te[wrong, 0], X2s_te[wrong, 1],
               facecolors="none", edgecolors=DARK,
               linewidths=2, s=140, zorder=4, label="Misclassified")
    acc2 = accuracy_score(y2_test, preds)
    ax.set_title(f"{title} — {acc2*100:.1f}% acc (2 features)")
    ax.set_xlabel(feat_names[0]); ax.set_ylabel(feat_names[1])
    ax.legend(loc="upper left", fontsize=9, framealpha=0.7)

handles2 = [mpatches.Patch(color=c, label=n)
            for c, n in zip(PALETTE, iris.target_names)]
fig3.legend(handles=handles2, loc="lower center",
            ncol=3, frameon=False, fontsize=10,
            bbox_to_anchor=(0.5, -0.04))
fig3.tight_layout()
fig3.savefig("/mnt/user-data/outputs/fig3_decision_boundaries.png",
             dpi=150, bbox_inches="tight", facecolor=BG)
plt.close(fig3)

In [19]:
fig4, ax = plt.subplots(figsize=(7, 4))
models  = ["KNN (k=5)\n(all 4 features)",
           "Logistic Regression\n(all 4 features)"]
accs    = [accuracy_score(y_test, y_pred_knn) * 100,
           accuracy_score(y_test, y_pred_lr)  * 100]
bars = ax.bar(models, accs, color=PALETTE[:2], width=0.45,
              edgecolor="white", linewidth=1.2)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=12, fontweight="bold")
ax.set_ylim(0, 115)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model Accuracy Comparison", fontweight="bold")
ax.axhline(100, linestyle="--", color="#aaa", linewidth=0.8)
fig4.tight_layout()
fig4.savefig("/mnt/user-data/outputs/fig4_accuracy_comparison.png",
             dpi=150, bbox_inches="tight", facecolor=BG)
plt.close(fig4)

print("\n  All 4 figures saved to /mnt/user-data/outputs/")


  All 4 figures saved to /mnt/user-data/outputs/
